# Part II — From Sequences to Transformers

## Recap: Where We Left Off

In Part I we built a complete pipeline:

- **BPE tokenization** — raw text → subword token IDs (open vocabulary)
- **Embeddings** — token IDs → dense vectors that carry semantic meaning
- **MLP** — embeddings → softmax → probability over next token

This pipeline works. But it treats its input as an **unordered bag of tokens**. "The cat sat" and "sat cat the" produce the same prediction. The model has no concept of sequence, order, or structure.

That is the problem we solve in this notebook.

---

## The Road Ahead

| Chapter | Concept | Problem Solved |
|---------|---------|----------------|
| 7 | RNN | Process tokens in order (but forget early context) |
| 8 | LSTM | Better long-term memory (but sequential bottleneck) |
| 9 | Attention | Look at all positions directly (but still uses RNN) |
| 10 | Transformer | Everything at once: parallel, deep, contextual |
| 11 | Generation | Turn the trained model into a text generator |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
sys.path.append('..')

from llmkit import data, viz
from llmkit.solutions import (
    ch06_bpe,
    ch07_rnn,
    ch08_lstm,
    ch09_attention,
    ch10_transformer,
    ch11_generation,
)

torch.manual_seed(42)
np.random.seed(42)

# Download and prepare WikiText-2
print("Loading corpus...")
corpus = data.get_small_corpus(n_sentences=300)
print(f"Corpus length: {len(corpus.split())} words")

# Train BPE tokenizer (re-use from Chapter 6)
print("Training BPE tokenizer...")
tokenizer = ch06_bpe.BytePairEncoding(vocab_size=1000)
tokenizer.train(corpus)
print(f"Vocabulary size: {len(tokenizer.vocab)} tokens")

# Encode the corpus to token IDs
token_ids = data.get_token_ids(corpus, tokenizer)
VOCAB_SIZE = len(tokenizer.vocab)
print(f"Total tokens: {len(token_ids)}, Vocab size: {VOCAB_SIZE}")
print("\nSetup complete!")

---

## The PyTorch Transition

> We built backpropagation by hand. We understand how gradients flow through a network. From here, the new ideas are in the *architectures* themselves — not the gradient plumbing. We let PyTorch handle what we've already proven we understand, so we can focus on what's new.

| Chapters | Tensors | Forward Pass | Backward Pass |
|----------|---------|-------------|---------------|
| 1–6 | `np.array` | Handwritten | Manual gradients + updates |
| 7–11 | `torch.Tensor` | **Handwritten** | `loss.backward()` + `optimizer.step()` |

The forward pass stays ours. The backward pass becomes automatic.


---

# Chapter 7: Recurrent Neural Network

**Track:** Prediction  
**Question:** How do we make the model aware of sequence and order?

---

## 7.1 Sliding Window Data Preparation

Before we can train on sequences, we need to frame language modeling as a supervised task. Given a flat list of token IDs, we slide a window of length $T$ across it. Each window becomes one training example: the input is the window, the target is the window shifted one step to the right.

The key insight is that the same token appears both as input and as target — depending on its position. This is **unsupervised** text turned into **supervised** training data, no human labeling needed.

---

$$\text{input}_i = [t_i, t_{i+1}, \ldots, t_{i+T-1}], \qquad \text{target}_i = [t_{i+1}, t_{i+2}, \ldots, t_{i+T}]$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $t_i$ | Token at position $i$ | Integer index into vocabulary |
| $T$ | Sequence length | Window size |
| $\text{input}_i$ | Input sequence | What the model sees |
| $\text{target}_i$ | Target sequence | What the model must predict, shifted by 1 |


## 7.1 Exercise

Implement the sliding window sequence builder.

In [ ]:
def your_prepare_sequences(token_ids, seq_len=32):
    """Create (input, target) pairs by sliding a window over token_ids.

    Each input[i] = token_ids[i : i+seq_len]
    Each target[i] = token_ids[i+1 : i+seq_len+1]  (shifted right by 1)
    """
    if not isinstance(token_ids, torch.Tensor):
        token_ids = torch.tensor(token_ids, dtype=torch.long)

    inputs, targets = [], []
    for i in range(len(token_ids) - seq_len):
        # TODO: Append the input window starting at i
        inputs.append(___)
        # TODO: Append the target window starting at i+1
        targets.append(___)

    return torch.stack(inputs), torch.stack(targets)

# Quick check
test_ids = torch.arange(20)
try:
    inp, tgt = your_prepare_sequences(test_ids, seq_len=5)
    print(f"inputs shape:  {inp.shape}   (expected: (15, 5))")
    print(f"targets shape: {tgt.shape}   (expected: (15, 5))")
    print(f"inp[0]: {inp[0].tolist()}")
    print(f"tgt[0]: {tgt[0].tolist()}  (should be inp[0] shifted right by 1)")
except Exception:
    print("[NOT IMPLEMENTED]")
    inp, tgt = ..., ...

## 7.1 Verify & Compare

In [ ]:
ref_inp, ref_tgt = ch07_rnn.prepare_sequences(test_ids, seq_len=5)

try:
    your_inp, your_tgt = your_prepare_sequences(test_ids, seq_len=5)
    viz.compare(your_inp.numpy(), ref_inp.numpy())
    viz.compare(your_tgt.numpy(), ref_tgt.numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

# Build dataset for training (use reference from now on)
SEQ_LEN = 32
inputs, targets = ch07_rnn.prepare_sequences(token_ids, seq_len=SEQ_LEN)
print(f"\nDataset: {len(inputs)} sequences of length {SEQ_LEN}")

## 7.2 The RNN Step

An RNN processes one token at a time. At each step $t$, it takes the current token embedding $x_t$ and the previous hidden state $h_{t-1}$, and produces a new hidden state $h_t$ that summarizes everything seen so far.

The hidden state is the RNN's **memory**. It carries information from position 1 all the way to position $T$. Each step writes to memory (via $W_{hh}$) and reads the current input (via $W_{xh}$).

---

$$h_t = \tanh\left(x_t W_{xh} + h_{t-1} W_{hh} + b_h\right)$$

---

| Symbol | Name | Shape |
|--------|------|---------|
| $x_t$ | Current token embedding | $(\text{batch}, d_{\text{embed}})$ |
| $h_{t-1}$ | Previous hidden state | $(\text{batch}, d_{\text{hidden}})$ |
| $W_{xh}$ | Input-to-hidden weights | $(d_{\text{embed}}, d_{\text{hidden}})$ |
| $W_{hh}$ | Hidden-to-hidden weights | $(d_{\text{hidden}}, d_{\text{hidden}})$ |
| $b_h$ | Bias | $(d_{\text{hidden}},)$ |
| $h_t$ | New hidden state | $(\text{batch}, d_{\text{hidden}})$ |


## 7.2 Exercise

Implement a single RNN timestep.

In [ ]:
def your_rnn_step(x_t, h_prev, W_xh, W_hh, b_h):
    """Single RNN timestep: h_t = tanh(x_t @ W_xh + h_prev @ W_hh + b_h)

    Args:
        x_t:    (batch, embed_dim)
        h_prev: (batch, hidden_size)
        W_xh:   (embed_dim, hidden_size)
        W_hh:   (hidden_size, hidden_size)
        b_h:    (hidden_size,)

    Returns:
        h_t: (batch, hidden_size)
    """
    # TODO: Linear combination of input and previous hidden state
    linear = ___
    # TODO: Apply tanh
    h_t = ___
    return h_t

# Quick shape check
batch, embed_dim, hidden_size = 4, 16, 32
x_test    = torch.randn(batch, embed_dim)
h_test    = torch.zeros(batch, hidden_size)
W_xh_test = torch.randn(embed_dim, hidden_size) * 0.01
W_hh_test = torch.randn(hidden_size, hidden_size) * 0.01
b_h_test  = torch.zeros(hidden_size)

try:
    h_out = your_rnn_step(x_test, h_test, W_xh_test, W_hh_test, b_h_test)
    if h_out is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"h_t shape: {h_out.shape}  (expected: ({batch}, {hidden_size}))")
        print(f"Values in [-1, 1]: {h_out.abs().max().item():.4f} (tanh bound)")
except Exception:
    print("[NOT IMPLEMENTED]")
    h_out = ...

## 7.2 Verify & Compare

In [ ]:
ref_h_out = ch07_rnn.rnn_step(x_test, h_test, W_xh_test, W_hh_test, b_h_test)

try:
    your_h_out = your_rnn_step(x_test, h_test, W_xh_test, W_hh_test, b_h_test)
    viz.compare(your_h_out.detach().numpy(), ref_h_out.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 7.2 Visualize

In [ ]:
# Show hidden state norms over a short sequence
model_rnn = ch07_rnn.RNNLanguageModel(VOCAB_SIZE, embed_dim=64, hidden_size=128)

with torch.no_grad():
    sample_input = inputs[:1]                       # 1 sequence
    emb = model_rnn.embedding(sample_input)         # (1, seq_len, embed_dim)
    h = torch.zeros(1, 128)
    h_seq = []
    for t in range(emb.shape[1]):
        h = ch07_rnn.rnn_step(emb[:, t, :], h, model_rnn.W_xh, model_rnn.W_hh, model_rnn.b_h)
        h_seq.append(h[0].numpy())
    h_seq = np.array(h_seq)

viz.plot_hidden_states(h_seq)

## 7.3 Training an RNN

Training an RNN is the same loop as before — forward pass, compute loss, backward, update — but now the forward pass unrolls through time. PyTorch's autograd tracks gradients through the entire sequence automatically.

One important addition: **gradient clipping**. When gradients are multiplied across many timesteps, they can grow exponentially (exploding gradients) or shrink to zero (vanishing gradients). Clipping caps the gradient norm at a maximum value, preventing explosions. Vanishing gradients are a deeper architectural problem that LSTM addresses.

---

$$\text{if } \|g\| > \text{clip\_norm}: \quad g \leftarrow g \cdot \frac{\text{clip\_norm}}{\|g\|}$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $g$ | Gradient vector | All parameters flattened |
| $\|g\|$ | Gradient norm | Total magnitude |
| clip_norm | Maximum norm | Threshold; typically 1.0 |


## 7.3 Exercise

Complete the training loop for the RNN language model.

In [ ]:
def your_train_rnn_step(model, xb, yb, optimizer, criterion):
    """One training step for the RNN language model.

    Args:
        model:     RNNLanguageModel
        xb:        (batch, seq_len) input token ids
        yb:        (batch, seq_len) target token ids
        optimizer: PyTorch optimizer
        criterion: loss function (CrossEntropyLoss)

    Returns:
        loss: scalar float
    """
    # TODO: Forward pass — get logits from model
    logits, _ = ___

    # Reshape for cross-entropy: (batch*seq, vocab) vs (batch*seq,)
    bs, sl, v = logits.shape
    # TODO: Compute loss
    loss = criterion(___, ___)

    # TODO: Zero gradients, backward, clip, step
    ___
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    ___

    return loss.item()

print("Training step defined (fill in the blanks above to try it)")

## 7.3 Verify & Compare

In [ ]:
# Train RNN with reference implementation
print("Training RNN language model...")
rnn_model = ch07_rnn.RNNLanguageModel(VOCAB_SIZE, embed_dim=64, hidden_size=128)
rnn_model, rnn_losses = ch07_rnn.train_rnn(
    rnn_model, inputs, targets,
    n_epochs=5, batch_size=64, lr=1e-3, verbose=True
)
print(f"\nFinal loss: {rnn_losses[-1]:.4f}")

## 7.3 Visualize

In [ ]:
viz.plot_loss_curve(rnn_losses, xlabel="Epoch")

**Key Takeaway:**
> We can process sequences now, but the model's memory fades with distance.

---

# Chapter 8: Long Short-Term Memory (LSTM)

**Track:** Prediction  
**Question:** Can we give the network a better mechanism for long-term memory?

---

## 8.1 Cell State and Gates

The RNN's hidden state does two things at once: remember the past and contribute to the current prediction. This dual role causes the vanishing gradient problem — when backpropagating through many tanh activations, gradients shrink exponentially.

The LSTM separates these concerns with a **cell state** $c_t$ — a dedicated memory lane that can carry information across hundreds of timesteps with minimal interference. Three **gates** (sigmoid networks that output values between 0 and 1) control what flows in, what flows out, and what gets erased.

---

$$f_t = \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i) \quad \text{(input gate)}$$
$$g_t = \tanh(x_t W_{xg} + h_{t-1} W_{hg} + b_g) \quad \text{(cell candidate)}$$
$$o_t = \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o) \quad \text{(output gate)}$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $f_t$ | Forget gate | 0 = erase, 1 = keep (from cell state) |
| $i_t$ | Input gate | 0 = ignore, 1 = write (new information) |
| $g_t$ | Cell candidate | What *could* be written to cell state |
| $o_t$ | Output gate | What part of cell state to expose as $h_t$ |
| $\sigma$ | Sigmoid | Squashes to $(0, 1)$ — acts as a soft switch |


## 8.1 Exercise

Implement the forget and input gate computations.

In [ ]:
def your_forget_gate(x_t, h_prev, W_xf, W_hf, b_f):
    """Forget gate: f_t = sigmoid(x_t @ W_xf + h_prev @ W_hf + b_f)"""
    # TODO: Compute forget gate activation
    return ___

def your_input_gate(x_t, h_prev, W_xi, W_hi, b_i):
    """Input gate: i_t = sigmoid(x_t @ W_xi + h_prev @ W_hi + b_i)"""
    # TODO: Compute input gate activation
    return ___

# Test
x_t   = torch.randn(4, 16)
h_prev = torch.randn(4, 32)
W_xf  = torch.randn(16, 32) * 0.01
W_hf  = torch.randn(32, 32) * 0.01
b_f   = torch.zeros(32)
W_xi, W_hi, b_i = W_xf.clone(), W_hf.clone(), b_f.clone()

try:
    f_t = your_forget_gate(x_t, h_prev, W_xf, W_hf, b_f)
    i_t = your_input_gate(x_t, h_prev, W_xi, W_hi, b_i)
    if f_t is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"Forget gate shape: {f_t.shape}, values in [0,1]: {f_t.min().item():.3f} – {f_t.max().item():.3f}")
        print(f"Input  gate shape: {i_t.shape}, values in [0,1]: {i_t.min().item():.3f} – {i_t.max().item():.3f}")
except Exception:
    print("[NOT IMPLEMENTED]")
    f_t, i_t = ..., ...

## 8.1 Verify & Compare

In [ ]:
ref_f_t = torch.sigmoid(x_t @ W_xf + h_prev @ W_hf + b_f)

try:
    your_f_t = your_forget_gate(x_t, h_prev, W_xf, W_hf, b_f)
    viz.compare(your_f_t.detach().numpy(), ref_f_t.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 8.2 The LSTM Step

The four gates work together in two steps. First, the cell state is updated: forget some of the old state, write some new candidate values. Second, the hidden state is derived from the updated cell state — filtered by the output gate.

The key property: the cell state update $c_t = f_t \odot c_{t-1} + i_t \odot g_t$ is a **linear combination** of $c_{t-1}$ and $g_t$ (no activation squashing). Gradients can flow back through time along this path without shrinking, which is why LSTMs train effectively over long sequences.

---

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t$$
$$h_t = o_t \odot \tanh(c_t)$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $c_t$ | Cell state | Long-term memory highway |
| $\odot$ | Element-wise multiply | Each gate independently controls each unit |
| $f_t \odot c_{t-1}$ | What to keep | Selectively erasing old memories |
| $i_t \odot g_t$ | What to add | New information weighted by input gate |
| $h_t$ | Hidden state | Short-term state exposed to the output head |


## 8.2 Exercise

Implement the full LSTM step.

In [ ]:
def your_lstm_step(x_t, h_prev, c_prev, W_x, W_h, b):
    """Single LSTM timestep.

    W_x, W_h, b are stacked for all 4 gates along dim=-1:
    [forget | input | gate | output] — each of size hidden_size.

    Args:
        x_t:    (batch, embed_dim)
        h_prev: (batch, hidden_size)
        c_prev: (batch, hidden_size)
        W_x:    (embed_dim, 4*hidden_size)
        W_h:    (hidden_size, 4*hidden_size)
        b:      (4*hidden_size,)

    Returns:
        h_t: (batch, hidden_size)
        c_t: (batch, hidden_size)
    """
    # Combined linear map for all gates
    gates = x_t @ W_x + h_prev @ W_h + b   # (batch, 4*hidden_size)
    hidden_size = h_prev.shape[1]

    # TODO: Split into 4 gates and apply activations
    f = torch.sigmoid(gates[:, 0 * hidden_size : 1 * hidden_size])   # forget
    i = ___   # input gate (sigmoid)
    g = ___   # cell candidate (tanh)
    o = ___   # output gate (sigmoid)

    # TODO: Update cell state
    c_t = ___
    # TODO: Compute new hidden state
    h_t = ___
    return h_t, c_t

# Shape check
batch, embed, hid = 4, 16, 32
x_t    = torch.randn(batch, embed)
h_prev = torch.zeros(batch, hid)
c_prev = torch.zeros(batch, hid)
W_x    = torch.randn(embed, 4 * hid) * 0.01
W_h    = torch.randn(hid,  4 * hid) * 0.01
b      = torch.zeros(4 * hid)

try:
    h_t, c_t = your_lstm_step(x_t, h_prev, c_prev, W_x, W_h, b)
    if h_t is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"h_t shape: {h_t.shape},  c_t shape: {c_t.shape}")
except Exception:
    print("[NOT IMPLEMENTED]")
    h_t, c_t = ..., ...

## 8.2 Verify & Compare

In [ ]:
ref_h_t, ref_c_t = ch08_lstm.lstm_step(x_t, h_prev, c_prev, W_x, W_h, b)

try:
    your_h_t, your_c_t = your_lstm_step(x_t, h_prev, c_prev, W_x, W_h, b)
    viz.compare(your_h_t.detach().numpy(), ref_h_t.detach().numpy())
    viz.compare(your_c_t.detach().numpy(), ref_c_t.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 8.3 Training the LSTM

The LSTM trains with the same loop as the RNN. The only difference is the forward pass — now two states are carried forward ($h_t$ and $c_t$) instead of one. PyTorch autograd traces through both.

We compare LSTM vs RNN loss to see the improvement directly.


## 8.3 Exercise

Train the LSTM and compare its loss curve to the RNN's.

In [ ]:
# Training is identical to RNN — only the model class changes.
# Fill in the blank: replace ___ with the correct model class from ch08_lstm.
YOUR_MODEL_CLASS = ___   # hint: ch08_lstm.???

try:
    your_lstm = YOUR_MODEL_CLASS(VOCAB_SIZE, embed_dim=64, hidden_size=128)
    print("Model created:", your_lstm.__class__.__name__)
except (TypeError, AttributeError):
    print("[NOT IMPLEMENTED] Replace ___ with the correct model class")

## 8.3 Verify & Compare

In [ ]:
lstm_model = ch08_lstm.LSTMLanguageModel(VOCAB_SIZE, embed_dim=64, hidden_size=128)
lstm_model, lstm_losses = ch08_lstm.train_lstm(
    lstm_model, inputs, targets,
    n_epochs=5, batch_size=64, lr=1e-3, verbose=True
)
print(f"\nRNN final loss:  {rnn_losses[-1]:.4f}")
print(f"LSTM final loss: {lstm_losses[-1]:.4f}")

## 8.3 Visualize

In [ ]:
import matplotlib.pyplot as plt

# Side-by-side loss comparison
plt.figure(figsize=(8, 4))
plt.plot(rnn_losses,  label='RNN',  linewidth=2)
plt.plot(lstm_losses, label='LSTM', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RNN vs LSTM Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Gate activation heatmap on one example
with torch.no_grad():
    sample = inputs[:1]
    emb = lstm_model.embedding(sample)
    h, c = torch.zeros(1, 128), torch.zeros(1, 128)
    gates_dict = {'forget': [], 'input': [], 'gate': [], 'output': []}
    for t in range(emb.shape[1]):
        raw = emb[:, t, :] @ lstm_model.W_x + h @ lstm_model.W_h + lstm_model.b
        hs = 128
        gates_dict['forget'].append(torch.sigmoid(raw[:, 0*hs:1*hs])[0].numpy())
        gates_dict['input'].append( torch.sigmoid(raw[:, 1*hs:2*hs])[0].numpy())
        gates_dict['gate'].append(  torch.tanh(   raw[:, 2*hs:3*hs])[0].numpy())
        gates_dict['output'].append(torch.sigmoid(raw[:, 3*hs:4*hs])[0].numpy())
        h, c = ch08_lstm.lstm_step(emb[:, t, :], h, c,
                                   lstm_model.W_x, lstm_model.W_h, lstm_model.b)
    gates_dict = {k: np.array(v) for k, v in gates_dict.items()}

viz.plot_gate_activations(gates_dict)

**Key Takeaway:**
> We can remember longer, but the sequential architecture and compressed memory are now the bottleneck.

---

# Chapter 9: Attention Mechanism

**Track:** Prediction  
**Question:** What if the model could look back at all previous positions and choose what's relevant right now?

---

## 9.1 Scaled Dot-Product Attention

The LSTM compresses all history into a single fixed-size vector. When predicting the verb, we might need to look back 15 tokens to the subject — but that information is diluted in the hidden state.

Attention replaces the bottleneck: at each prediction step, compute a **relevance score** between the current position and every past position, convert to weights, and take a weighted sum of all past values. The model learns which positions to attend to — it can directly reach back to any token, regardless of distance.

---

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $Q$ | Query | "What am I looking for?" — projection of current position |
| $K$ | Key | "What do I contain?" — projection of each candidate position |
| $V$ | Value | "What information do I carry?" — projection of each position's content |
| $QK^T$ | Raw scores | How relevant is each key to this query? |
| $\sqrt{d_k}$ | Scaling | Prevents saturation in softmax when $d_k$ is large |
| $d_k$ | Key dimension | Size of Q and K vectors |


## 9.1 Exercise

Implement scaled dot-product attention.

In [ ]:
def your_scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention.

    Args:
        Q:    (batch, q_len, d_k)
        K:    (batch, k_len, d_k)
        V:    (batch, k_len, d_v)
        mask: (batch, q_len, k_len) bool — True positions are set to -inf

    Returns:
        output:  (batch, q_len, d_v)
        weights: (batch, q_len, k_len)
    """
    d_k = Q.shape[-1]

    # TODO: Compute raw scores = Q @ K^T / sqrt(d_k)
    scores = ___

    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    # TODO: Softmax over last dimension (key positions)
    weights = ___

    # TODO: Weighted sum of values
    output = ___

    return output, weights

# Shape check
Q_test = torch.randn(2, 5, 16)   # batch=2, 5 query positions, d_k=16
K_test = torch.randn(2, 5, 16)   # same for keys
V_test = torch.randn(2, 5, 32)   # d_v=32

try:
    out, w = your_scaled_dot_product_attention(Q_test, K_test, V_test)
    if out is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"output shape:  {out.shape}   (expected: (2, 5, 32))")
        print(f"weights shape: {w.shape}    (expected: (2, 5, 5))")
        print(f"weights sum to 1: {w[0, 0].sum().item():.4f}")
except Exception:
    print("[NOT IMPLEMENTED]")
    out, w = ..., ...

## 9.1 Verify & Compare

In [ ]:
ref_out, ref_w = ch09_attention.scaled_dot_product_attention(Q_test, K_test, V_test)

try:
    your_out, your_w = your_scaled_dot_product_attention(Q_test, K_test, V_test)
    viz.compare(your_out.detach().numpy(), ref_out.detach().numpy())
    viz.compare(your_w.detach().numpy(), ref_w.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 9.1 Visualize

In [ ]:
# Show attention weights for a short sample sequence
sample_ids = tokenizer.encode_ids("the cat sat on the mat")[:8]
sample_tensor = torch.tensor(sample_ids).unsqueeze(0)  # (1, seq)

attn_layer = ch09_attention.AttentionLayer(hidden_size=128, attention_dim=64)

# Use LSTM hidden states as input to attention
with torch.no_grad():
    emb = lstm_model.embedding(sample_tensor)
    h, c = torch.zeros(1, 128), torch.zeros(1, 128)
    h_list = []
    for t in range(emb.shape[1]):
        h, c = ch08_lstm.lstm_step(emb[:, t, :], h, c,
                                   lstm_model.W_x, lstm_model.W_h, lstm_model.b)
        h_list.append(h)
    h_seq = torch.stack(h_list, dim=1)   # (1, seq, 128)
    _, weights = attn_layer(h_seq)       # (1, seq, seq)

token_labels = [tokenizer.decode_ids([t]) for t in sample_ids]
viz.plot_attention_weights(weights[0].numpy(), token_labels)

## 9.2 Training with Attention

We add attention on top of the LSTM. The LSTM still processes tokens sequentially, but instead of predicting from the final hidden state alone, attention aggregates information from all hidden states into a context-aware representation at each position.

The sequential bottleneck is still here — we can't parallelize the LSTM. Attention is the tool; removing the LSTM entirely is what the Transformer does.


## 9.2 Exercise

Fill in the forward pass of `RNNWithAttention`.

In [ ]:
# The RNNWithAttention model in ch09_attention.py already exists.
# Here we sketch the key piece: applying attention after the LSTM.

def your_attention_forward(h_seq, attention_layer):
    """Apply attention to a sequence of hidden states.

    Args:
        h_seq:           (batch, seq_len, hidden_size)
        attention_layer: AttentionLayer instance

    Returns:
        context: (batch, seq_len, hidden_size) — attended representation
        weights: (batch, seq_len, seq_len)
    """
    # TODO: Call attention_layer on h_seq
    context, weights = ___
    return context, weights

print("Sketch defined — implement your_attention_forward above")

## 9.2 Verify & Compare

In [ ]:
print("Training LSTM + Attention model...")
attn_model = ch09_attention.RNNWithAttention(VOCAB_SIZE, embed_dim=64, hidden_size=128)
attn_model, attn_losses = ch09_attention.train_attention(
    attn_model, inputs, targets,
    n_epochs=5, batch_size=64, lr=1e-3, verbose=True
)
print(f"\nLSTM final loss:         {lstm_losses[-1]:.4f}")
print(f"LSTM+Attention final loss: {attn_losses[-1]:.4f}")

## 9.2 Visualize

In [ ]:
viz.plot_loss_curve(attn_losses, xlabel="Epoch")

**Key Takeaway:**
> Attention lets the model look anywhere in the sequence, but the sequential backbone underneath is holding us back.

---

# Chapter 10: The Transformer

**Track:** Prediction + Representation  
**Question:** What if we throw away the RNN entirely and use *only* attention?

---

## 10.1 Causal Masking and Positional Encoding

Without recurrence, the transformer has no inherent sense of order — "cat sat the" and "the cat sat" would be identical without positional information. We inject position by adding a **positional encoding** vector to each token embedding before processing.

We also need a **causal mask** to prevent cheating during training: when predicting token 5, the model must not see tokens 6, 7, 8. We block future positions by adding $-\infty$ to those attention scores before softmax (they become 0 after softmax).

---

$$\text{PE}(\text{pos}, 2i) = \sin\!\left(\frac{\text{pos}}{10000^{2i/d}}\right), \quad \text{PE}(\text{pos}, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{2i/d}}\right)$$

$$\text{mask}[i,j] = \begin{cases} -\infty & j > i \\ 0 & j \leq i \end{cases}$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| pos | Position | Token index in the sequence (0, 1, 2, …) |
| $i$ | Dimension index | Which pair of encoding dimensions |
| $d$ | Model dimension | Total embedding size |
| $10000^{2i/d}$ | Frequency denominator | Different frequencies for different dimensions |
| mask$[i,j]$ | Attention mask | Block position $j$ when predicting at position $i$ if $j > i$ |


## 10.1 Exercise

Implement the causal mask and positional encoding.

In [ ]:
def your_make_causal_mask(seq_len, device=None):
    """Upper-triangular causal mask.

    Returns a BoolTensor of shape (1, seq_len, seq_len).
    True  = masked (future) — will become -inf in attention scores.
    False = allowed (past or current).
    """
    # TODO: Create a seq_len x seq_len matrix of Trues, then zero out
    #       the lower triangle (including diagonal) — keep only upper triangle.
    # Hint: torch.ones(..., dtype=torch.bool) then torch.triu(..., diagonal=1)
    mask = ___
    return mask.unsqueeze(0)   # add batch dimension


def your_positional_encoding(seq_len, d_model, device=None):
    """Sinusoidal positional encoding. Returns (1, seq_len, d_model)."""
    import math
    pe  = torch.zeros(seq_len, d_model, device=device)
    pos = torch.arange(seq_len, dtype=torch.float, device=device).unsqueeze(1)
    # TODO: Compute the frequency denominator for even dimensions
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float, device=device)
        * ___
    )
    # TODO: Fill sin for even dimensions and cos for odd
    pe[:, 0::2] = ___
    pe[:, 1::2] = ___
    return pe.unsqueeze(0)

# Tests
try:
    m = your_make_causal_mask(5)
    pe = your_positional_encoding(10, 16)
    print(f"Mask shape: {m.shape}")
    print(f"Mask (5x5):\n{m[0].int()}")
    print(f"PE shape: {pe.shape}")
except Exception:
    print("[NOT IMPLEMENTED]")
    m = ...
    pe = ...

## 10.1 Verify & Compare

In [ ]:
ref_mask = ch10_transformer.make_causal_mask(5)
ref_pe   = ch10_transformer.positional_encoding(10, 16)

try:
    your_mask = your_make_causal_mask(5)
    viz.compare(your_mask.numpy().astype(int), ref_mask.numpy().astype(int))
    your_pe = your_positional_encoding(10, 16)
    viz.compare(your_pe.numpy(), ref_pe.numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 10.1 Visualize

In [ ]:
pe_display = ch10_transformer.positional_encoding(64, 32)
viz.plot_positional_encoding(pe_display)

## 10.2 Multi-Head Attention

Running attention once gives one pattern of relationships. Multi-head attention runs $h$ parallel attention operations — each with its own Q, K, V projections — then concatenates and projects the results.

Why? Different heads learn different relationship types. One head might learn subject-verb agreement. Another might learn pronoun-antecedent resolution. A third might capture local bigram patterns. They operate at dimension $d_{\text{model}}/h$ each, keeping total computation constant.

---

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW^Q_i, KW^K_i, VW^V_i)$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $h$ | Number of heads | Parallel attention operations |
| $d_{\text{head}} = d_{\text{model}} / h$ | Head dimension | Each head sees a fraction of the full representation |
| $W^Q_i, W^K_i, W^V_i$ | Head projections | Each head has its own learned projection |
| $W^O$ | Output projection | Mixes the concatenated head outputs back to $d_{\text{model}}$ |


## 10.2 Exercise

Implement the multi-head attention forward pass.

In [ ]:
class YourMultiHeadAttention(nn.Module):
    """Multi-head self-attention."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape

        # Project and split into heads: (batch, n_heads, seq, d_head)
        # TODO: Project x → Q, K, V, then reshape to (batch, n_heads, seq, d_head)
        Q = self.W_q(x).view(batch, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        K = ___
        V = ___

        # Reshape to (batch*n_heads, seq, d_head) for reuse of our attention primitive
        bh = batch * self.n_heads
        Q_ = Q.contiguous().view(bh, seq_len, self.d_head)
        K_ = K.contiguous().view(bh, seq_len, self.d_head)
        V_ = V.contiguous().view(bh, seq_len, self.d_head)

        if mask is not None:
            mask_ = mask.expand(batch, self.n_heads, seq_len, seq_len) \
                        .contiguous().view(bh, seq_len, seq_len)
        else:
            mask_ = None

        # TODO: Apply scaled dot-product attention (use ch09_attention.scaled_dot_product_attention)
        context, weights_flat = ___

        # Reassemble: (batch, seq, d_model) then project
        context = context.view(batch, self.n_heads, seq_len, self.d_head)
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        output  = self.W_o(context)
        weights = weights_flat.view(batch, self.n_heads, seq_len, seq_len)
        return output, weights

# Test
d_model, n_heads, seq = 32, 4, 8
mha_test = YourMultiHeadAttention(d_model, n_heads)
x_test   = torch.randn(2, seq, d_model)

try:
    out_test, w_test = mha_test(x_test)
    print(f"Output shape:  {out_test.shape}   (expected: (2, {seq}, {d_model}))")
    print(f"Weights shape: {w_test.shape} (expected: (2, {n_heads}, {seq}, {seq}))")
except Exception as e:
    print(f"[NOT IMPLEMENTED] {e}")

## 10.2 Verify & Compare

In [ ]:
ref_mha = ch10_transformer.MultiHeadAttention(d_model, n_heads)
# Copy weights so we compare same parameters
ref_mha.load_state_dict(mha_test.state_dict())

try:
    ref_out, ref_w = ref_mha(x_test)
    your_out, your_w = mha_test(x_test)
    viz.compare(your_out.detach().numpy(), ref_out.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 10.3 Transformer Block

A transformer block wraps multi-head attention and a feed-forward network in two **residual connections** with **layer normalization**. This pairing is what makes it possible to stack dozens of blocks without gradients vanishing.

The residual connection $x + \text{SubLayer}(x)$ creates a direct gradient path: even if the sublayer contributes nothing, the gradient still flows through $x$ unchanged. Layer normalization stabilizes the scale of activations at each layer, allowing higher learning rates and faster convergence.

---

$$x \leftarrow \text{LayerNorm}(x + \text{MultiHeadAttention}(x))$$
$$x \leftarrow \text{LayerNorm}(x + \text{FFN}(x))$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $x + \text{Sub}(x)$ | Residual connection | Skip-path so gradients can bypass the sublayer |
| LayerNorm | Layer normalization | Normalize activations across features; stabilizes training |
| FFN | Feed-forward network | Two linear layers with GELU; processes each position independently |


## 10.3 Exercise

Implement the transformer block forward pass.

In [ ]:
class YourTransformerBlock(nn.Module):
    """Single transformer decoder block."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn  = ch10_transformer.MultiHeadAttention(d_model, n_heads)
        self.ff    = ch10_transformer.FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, d_model)
        Returns: x (same shape), weights (batch, n_heads, seq, seq)
        """
        # TODO: Self-attention + residual + norm
        attn_out, weights = self.attn(x, mask)
        x = ___

        # TODO: Feed-forward + residual + norm
        x = ___

        return x, weights

# Test
block_test = YourTransformerBlock(d_model=32, n_heads=4)
x_b = torch.randn(2, 8, 32)

try:
    out_b, w_b = block_test(x_b)
    print(f"Block output shape: {out_b.shape}   (expected: (2, 8, 32))")
except Exception as e:
    print(f"[NOT IMPLEMENTED] {e}")

## 10.3 Verify & Compare

In [ ]:
ref_block = ch10_transformer.TransformerBlock(d_model=32, n_heads=4)
ref_block.load_state_dict(block_test.state_dict())

try:
    ref_out_b, _ = ref_block(x_b)
    your_out_b, _ = block_test(x_b)
    viz.compare(your_out_b.detach().numpy(), ref_out_b.detach().numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

## 10.4 The Full Transformer Language Model

We stack $N$ transformer blocks. The initial token embedding is looked up and added to the positional encoding. The causal mask is applied at every block. After the final block, a linear output head projects to vocabulary logits.

The grand payoff: as self-attention mixes information between all positions, the representation of "bank" in "river bank" diverges from "bank" in "bank account" — **contextual embeddings emerge automatically** as a byproduct of the prediction architecture. The two tracks converge here.


## 10.4 Exercise

Identify the key steps in the transformer forward pass.

In [ ]:
# The full model is in ch10_transformer.TransformerLanguageModel.
# Trace through its forward method to answer these questions:

t_model = ch10_transformer.TransformerLanguageModel(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=2
)

x_sample = inputs[:4]   # 4 sequences
with torch.no_grad():
    logits, all_weights = t_model(x_sample)

# Fill in the expected shapes:
print(f"Input shape:        {x_sample.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"Number of layers:   {len(all_weights)}")
print(f"Weights[0] shape:   {all_weights[0].shape}")
print()
print(f"Expected logits shape: (4, {SEQ_LEN}, {VOCAB_SIZE})")

## 10.4 Verify & Compare

In [ ]:
print("Training Transformer language model...")
transformer = ch10_transformer.TransformerLanguageModel(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=2
)
transformer, tf_losses = ch10_transformer.train_transformer(
    transformer, inputs, targets,
    n_epochs=10, batch_size=64, lr=3e-4, verbose=True
)
print(f"\nRNN final loss:         {rnn_losses[-1]:.4f}")
print(f"LSTM final loss:        {lstm_losses[-1]:.4f}")
print(f"Transformer final loss: {tf_losses[-1]:.4f}")

## 10.4 Visualize

In [ ]:
viz.plot_loss_curve(tf_losses, xlabel="Epoch")

# Multi-head attention patterns on a short sequence
sample_ids = tokenizer.encode_ids("the cat sat on the mat")[:8]
s_tensor = torch.tensor(sample_ids).unsqueeze(0)

with torch.no_grad():
    _, all_w = transformer(s_tensor)

token_labels = [tokenizer.decode_ids([t]) for t in sample_ids]
# Show first layer's weights, removing batch dim
viz.plot_multi_head_attention(all_w[0][0].numpy(), token_labels)

**Key Takeaway:**
> The transformer processes all tokens in parallel, enables arbitrary-depth context, and produces contextual embeddings as a side effect. Both tracks converge here.

---

# Chapter 11: Autoregressive Generation

**Track:** Prediction (at inference time)  
**Question:** We have a trained model. How do we actually generate text?

---

## 11.1 The Autoregressive Loop

During training, we feed the whole sequence at once and compute loss at every position in parallel. At inference time, we don't have the future — we generate it.

The process is **autoregressive**: feed the prompt, get the next-token distribution, sample a token, append it to the input, repeat. Each generated token becomes part of the context for the next step. We are running the model one token at a time.

---

$$\text{ids}_{t+1} = [\text{ids}_0, \ldots, \text{ids}_t, \hat{t}_{t+1}], \quad \hat{t}_{t+1} \sim p(\cdot \mid \text{ids}_0, \ldots, \text{ids}_t)$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $\text{ids}_t$ | Token sequence at step $t$ | All tokens generated so far |
| $p(\cdot \mid \ldots)$ | Conditional distribution | Model output: softmax over vocabulary |
| $\hat{t}_{t+1}$ | Sampled next token | One token drawn from the distribution |
| argmax | Greedy decoding | Always pick the most likely token |


## 11.1 Exercise

Implement greedy decoding.

In [ ]:
def your_greedy_decode(model, prompt_ids, max_new_tokens=30):
    """Generate tokens greedily — always pick the highest-probability next token.

    Args:
        model:          TransformerLanguageModel
        prompt_ids:     LongTensor (seq_len,)
        max_new_tokens: How many tokens to generate

    Returns:
        LongTensor (seq_len + max_new_tokens,)
    """
    model.eval()
    ids = prompt_ids.clone().unsqueeze(0)   # (1, seq)
    max_ctx = model.max_seq_len

    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = ids[:, -max_ctx:]
            # TODO: Forward pass — get logits
            logits, _ = ___
            # TODO: Get logits at the last position
            next_logits = ___
            # TODO: Pick the argmax
            next_id = ___
            ids = torch.cat([ids, next_id.unsqueeze(0).unsqueeze(0)], dim=1)

    return ids[0]

# Quick check
prompt_tokens = tokenizer.encode_ids("the")
prompt_tensor = torch.tensor(prompt_tokens, dtype=torch.long)

try:
    generated = your_greedy_decode(transformer, prompt_tensor, max_new_tokens=10)
    if generated is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"Generated {len(generated) - len(prompt_tokens)} new tokens")
        print(tokenizer.decode_ids(generated.tolist()))
except Exception:
    print("[NOT IMPLEMENTED]")
    generated = ...

## 11.1 Verify & Compare

In [ ]:
ref_generated = ch11_generation.greedy_decode(transformer, prompt_tensor, max_new_tokens=10)

try:
    your_generated = your_greedy_decode(transformer, prompt_tensor, max_new_tokens=10)
    viz.compare(your_generated.numpy(), ref_generated.numpy())
except Exception:
    print("[NOT IMPLEMENTED]")

print("\nGreedy output:", tokenizer.decode_ids(ref_generated.tolist()))

## 11.2 Temperature and Top-k Sampling

Greedy decoding always picks the most likely token. The result is safe but repetitive — the model circles through common phrases. Sampling introduces randomness, producing more varied and natural text.

**Temperature** divides all logits by $T$ before softmax. Low temperature ($T < 1$) sharpens the distribution — the top token dominates even more. High temperature ($T > 1$) flattens it — more tokens become equally likely.

**Top-k** restricts sampling to the $k$ most probable tokens, filtering out low-probability noise. Without it, very low-probability tokens ("aardvark" appearing as next token after "the") can occasionally be sampled with disastrous results.

---

$$p_i^{(T)} = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}, \quad \text{top-k: set } p_i = 0 \text{ for all but the top-}k \text{ tokens}$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $T$ | Temperature | Controls distribution sharpness |
| $T < 1$ | Low temperature | More confident, less diverse |
| $T > 1$ | High temperature | More random, more creative |
| $k$ | Top-k | Number of tokens kept before sampling |


## 11.2 Exercise

Implement top-k sampling.

In [ ]:
def your_top_k_sample(logits, k, temperature=1.0):
    """Sample from the top-k most probable tokens.

    Args:
        logits:      (vocab_size,)
        k:           Number of tokens to keep
        temperature: Scale factor before softmax

    Returns:
        token_id: int
    """
    k = min(k, logits.shape[-1])
    # TODO: Find the k-th largest logit value (threshold)
    top_k_values, _ = ___
    threshold = ___

    # TODO: Zero out everything below threshold
    filtered = logits.clone()
    filtered[___] = float('-inf')

    # TODO: Apply temperature and sample
    scaled = filtered / temperature
    probs  = F.softmax(scaled, dim=-1)
    return ___

# Quick check
test_logits = torch.randn(VOCAB_SIZE)
try:
    token_id = your_top_k_sample(test_logits, k=10)
    if token_id is ...:
        print("[NOT IMPLEMENTED]")
    else:
        print(f"Sampled token id: {token_id}  (valid: {0 <= token_id < VOCAB_SIZE})")
except Exception:
    print("[NOT IMPLEMENTED]")

## 11.2 Verify & Compare

In [ ]:
# Verify statistical behavior: top-k=1 should always return argmax
ref_greedy = test_logits.argmax().item()

try:
    deterministic_result = your_top_k_sample(test_logits, k=1, temperature=0.001)
    match = (deterministic_result == ref_greedy)
    print(f"[{'PASS' if match else 'FAIL'}] top_k=1 returns argmax: {deterministic_result} vs {ref_greedy}")
except Exception:
    print("[NOT IMPLEMENTED]")

## 11.3 Nucleus (Top-p) Sampling

Top-k uses a fixed number of tokens ($k = 40$ always keeps 40 tokens). This is inflexible: sometimes the top 3 tokens cover 99% of the probability mass; other times the distribution is flat and 40 tokens cover only 60%.

**Nucleus sampling** adapts: sort tokens by probability descending, take the smallest prefix whose cumulative probability exceeds $p$, sample only from that prefix. When the model is confident, the nucleus is small. When the model is uncertain, the nucleus expands automatically.

---

$$\text{nucleus}(p) = \text{argmin}_{S} \left\{S : \sum_{i \in S} p_i \geq p\right\}$$

---

| Symbol | Name | Meaning |
|--------|------|---------|
| $p$ | Nucleus threshold | Minimum cumulative probability to include |
| $S$ | Nucleus set | Smallest top-probability set summing to at least $p$ |
| Adaptive | Nucleus behavior | Small when confident, large when uncertain |


## 11.3 Exercise

Implement nucleus sampling and run the final generation comparison.

In [ ]:
def your_nucleus_sample(logits, p, temperature=1.0):
    """Nucleus (top-p) sampling.

    Args:
        logits:      (vocab_size,)
        p:           Cumulative probability threshold (0 < p <= 1)
        temperature: Scaling factor

    Returns:
        token_id: int
    """
    scaled = logits / temperature
    probs  = F.softmax(scaled, dim=-1)

    # TODO: Sort probabilities descending
    sorted_probs, sorted_indices = ___

    # TODO: Compute cumulative sum
    cumulative = ___

    # TODO: Build remove mask: True for tokens where cumulative − prob > p
    #       (shift by 1 so we keep at least the top token)
    remove_mask = ___

    sorted_probs[remove_mask] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum()   # renormalize
    sampled_idx  = torch.multinomial(sorted_probs, num_samples=1).item()
    return sorted_indices[sampled_idx].item()

try:
    nucleus_id = your_nucleus_sample(test_logits, p=0.9)
    print(f"Nucleus sample token id: {nucleus_id}")
except Exception:
    print("[NOT IMPLEMENTED]")

## 11.3 Verify & Compare

In [ ]:
# Compare generated text across strategies
prompt_text = "the history of"

samples = {}
for strategy, kwargs in [
    ('greedy',      dict(strategy='greedy')),
    ('temp=0.7',    dict(strategy='temperature', temperature=0.7)),
    ('top_k=40',    dict(strategy='top_k', k=40, temperature=0.9)),
    ('nucleus p=0.9', dict(strategy='nucleus', p=0.9, temperature=0.9)),
]:
    samples[strategy] = ch11_generation.generate(
        transformer, tokenizer, prompt_text, max_new_tokens=30, **kwargs
    )

for strategy, text in samples.items():
    print(f"[{strategy}]")
    print(f"  {text}")
    print()

## 11.3 Visualize

In [ ]:
viz.plot_generation_samples(samples)

**Key Takeaway:**
> Generation is a loop: predict one token, append, repeat. Temperature, top-k, and top-p control the trade-off between coherence and creativity.

---

# Part II Summary

## The Complete Arc

| Chapter | Concept | Problem Solved |
|---------|---------|----------------|
| 7 | RNN | Sequential processing — order matters now |
| 8 | LSTM | Gated memory — long-range dependencies |
| 9 | Attention | Direct access to any past position |
| 10 | Transformer | Parallel + deep + contextual embeddings |
| 11 | Generation | Prompt → text via autoregressive sampling |

## The Two-Track Convergence

- **Representation track:** One-hot → Word2Vec → BPE → **Contextual embeddings (Transformer)**
- **Prediction track:** Linear regression → Logistic → MLP → RNN → LSTM → Attention → **Transformer** → Generation

Both tracks converge at the Transformer. The prediction architecture solves the representation problem as a side effect: contextual embeddings emerge from self-attention without any separate representation learning step.

## What You Now Have

Every conceptual and practical piece needed to implement a working GPT-style language model — from tokenization to generation.
